# Pandas — łączenie i agregacja

**Programowanie w Pythonie II** | Wykład 8
**Politechnika Opolska** | Analityka danych w biznesie

---

## Co dzisiaj?

Dotąd pracowaliście na **jednej tabeli**. Ale w praktyce dane biznesowe są **rozproszone po wielu tabelach** — jedna ma klientów, druga zamówienia, trzecia produkty. Żeby odpowiedzieć na pytanie *„który klient kupił najwięcej?"* musisz te tabele **połączyć** — a potem **zagregować** wyniki.

To dokładnie to, co robi analityk każdego dnia.

```mermaid
graph TD
    A["Tabela 1 — Klienci"] --> M["merge — łączenie"]
    B["Tabela 2 — Zamówienia"] --> M
    C["Tabela 3 — Produkty"] --> M
    M --> G["groupby — grupowanie"]
    G --> P["pivot_table — agregacja 2D"]
    P --> D["Decyzja biznesowa"]
```

### Po tym wykładzie potrafisz:

1. **Łączyć** dwie lub więcej tabel używając `merge()` z odpowiednim typem złączenia (Bloom 3)
2. **Stosować** `concat()` do sklejania wierszy i kolumn (Bloom 3)
3. **Analizować** dane grupując je po kategorii z `groupby()` + `agg()` (Bloom 4)
4. **Tworzyć** tabele przestawne `pivot_table()` i liczebności `crosstab()` (Bloom 5)

> **Ciekawostka:** Operacje `merge` i `groupby` w Pandas są bezpośrednimi odpowiednikami polecenia `JOIN` i `GROUP BY` w SQL — języku baz danych. Jeśli kiedyś spotkasz się z SQL, 80% wiedzy z dzisiejszego wykładu jest przenośna. A `pivot_table()` to ta sama **tabela przestawna**, którą znasz z Excela — tylko programowalna.


## Import

Do dzisiejszego wykładu potrzebujemy tylko Pandas i NumPy — naszych dwóch głównych narzędzi.

In [ ]:
import pandas as pd
import numpy as np
print(f"Pandas wersja: {pd.__version__}")
print(f"NumPy wersja:  {np.__version__}")

---

## 1. Dane — AutoRent: wypożyczalnia samochodów

Pracujemy dziś na realistycznym zestawie **trzech tabel** z wypożyczalni samochodów „AutoRent". Każda tabela opisuje inny aspekt biznesu i każda ma **klucz** łączący ją z pozostałymi.

```mermaid
graph TD
    K["klienci<br/>klient_id (klucz)<br/>imię, nazwisko<br/>miasto, wiek, segment"]
    A["samochody<br/>auto_id (klucz)<br/>marka, model<br/>segment, cena_dzien"]
    W["wypozyczenia<br/>klient_id → klienci<br/>auto_id → samochody<br/>data_od, dni, stacja"]
    W -->|"klient_id"| K
    W -->|"auto_id"| A
```

**Słownik kluczy** — tak to często spotkasz w biznesie:
- `klienci.klient_id` jest **kluczem głównym** (primary key) — identyfikuje jednego klienta
- `samochody.auto_id` jest kluczem głównym — identyfikuje jeden samochód
- `wypozyczenia.klient_id` i `wypozyczenia.auto_id` są **kluczami obcymi** (foreign keys) — wskazują na klienta i samochód w pozostałych tabelach

### Tabela 1 — `klienci`

Podstawowe dane o naszych klientach. Segment (`premium` / `standard`) to ich klasyfikacja marketingowa — wykorzystamy ją w analizie.

In [ ]:
# Tabela 1: klienci
klienci = pd.DataFrame({
    'klient_id': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112],
    'imie':      ['Anna', 'Jan', 'Ewa', 'Marek', 'Kasia', 'Piotr',
                   'Olga', 'Tomasz', 'Zofia', 'Krzysztof', 'Maria', 'Paweł'],
    'nazwisko':  ['Kowalska', 'Nowak', 'Wiśniewska', 'Wójcik', 'Kowalczyk', 'Kamiński',
                   'Lewandowska', 'Szymański', 'Dąbrowska', 'Zieliński', 'Szewczyk', 'Woźniak'],
    'miasto':    ['Warszawa', 'Kraków', 'Warszawa', 'Gdańsk', 'Wrocław', 'Poznań',
                   'Warszawa', 'Kraków', 'Gdańsk', 'Wrocław', 'Warszawa', 'Kraków'],
    'wiek':      [34, 28, 45, 52, 31, 39, 26, 48, 55, 42, 33, 29],
    'segment':   ['premium', 'standard', 'premium', 'premium', 'standard', 'standard',
                   'standard', 'premium', 'premium', 'standard', 'standard', 'standard']
})
print(f"klienci: {klienci.shape[0]} wierszy, {klienci.shape[1]} kolumn")
klienci.head()

### Tabela 2 — `samochody`

Flota wypożyczalni. Zwróć uwagę na kolumnę `segment_auto` — to ta sama koncepcja co `segment` u klientów, ale dotyczy samochodów (nie pomylcie tych kolumn!).

In [ ]:
# Tabela 2: samochody — flota wypożyczalni
samochody = pd.DataFrame({
    'auto_id':    [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'marka':      ['Toyota', 'Toyota', 'BMW', 'BMW', 'Volkswagen', 'Volkswagen',
                    'Audi', 'Audi', 'Skoda', 'Skoda'],
    'model':      ['Yaris', 'Corolla', 'Series 3', 'X5', 'Golf', 'Passat',
                    'A4', 'Q5', 'Fabia', 'Octavia'],
    'segment_auto': ['ekonomiczny', 'ekonomiczny', 'premium', 'SUV', 'sredni', 'sredni',
                      'premium', 'SUV', 'ekonomiczny', 'sredni'],
    'rok':        [2022, 2023, 2022, 2023, 2022, 2023, 2023, 2024, 2022, 2023],
    'cena_dzien': [120, 150, 280, 420, 180, 220, 310, 450, 100, 200]
})
print(f"samochody: {samochody.shape[0]} wierszy")
samochody.head()

### Tabela 3 — `wypozyczenia`

Historia transakcji. Każdy wiersz to jedno wypożyczenie. Sama ta tabela jest prawie bezużyteczna — dopóki nie połączymy jej z tabelami `klienci` i `samochody`, to tylko lista numerów ID.

In [ ]:
# Tabela 3: wypozyczenia — historia transakcji
np.random.seed(42)
n = 60
wypozyczenia = pd.DataFrame({
    'wypoz_id':  range(1001, 1001 + n),
    'klient_id': np.random.choice(klienci['klient_id'], size=n),
    'auto_id':   np.random.choice(samochody['auto_id'], size=n),
    'data_od':   pd.to_datetime('2026-01-01') + pd.to_timedelta(np.random.randint(0, 90, n), unit='D'),
    'dni':       np.random.randint(1, 14, size=n),
    'stacja':    np.random.choice(['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław', 'Poznań'], size=n)
})
print(f"wypozyczenia: {wypozyczenia.shape[0]} wierszy")
wypozyczenia.head()

In [ ]:
# Podgląd wszystkich trzech tabel — sprawdzenie struktury
print(f"Klienci:       {klienci.shape}")
print(f"Samochody:     {samochody.shape}")
print(f"Wypożyczenia:  {wypozyczenia.shape}")
print(f"\nŁączniki (klucze):")
print(f"  wypozyczenia.klient_id -> klienci.klient_id")
print(f"  wypozyczenia.auto_id   -> samochody.auto_id")

**Analogia Excela:** Wyobraź sobie trzy arkusze w jednym pliku:
- Arkusz „Klienci" — baza klientów
- Arkusz „Samochody" — flota
- Arkusz „Wypożyczenia" — lista transakcji

W Excelu łączysz je formułą `WYSZUKAJ.PIONOWO()` / `XLOOKUP()`. W Pandas robisz to jedną funkcją: `merge()`.

---

## 2. `merge()` — łączenie dwóch tabel

`merge()` to **najważniejsza funkcja dzisiejszego wykładu**. Łączy dwie tabele po wspólnej kolumnie (kluczu).

Składnia:
```python
pd.merge(lewa, prawa, on='klucz', how='inner')
```

albo (częściej używane — metoda DataFrame):
```python
lewa.merge(prawa, on='klucz', how='inner')
```

In [ ]:
# Najprostszy merge: wypożyczenia + klienci po kluczu klient_id
wypoz_z_klientem = wypozyczenia.merge(klienci, on='klient_id')
print(f"Shape: {wypoz_z_klientem.shape}")
wypoz_z_klientem.head()

**Co się stało?** Pandas znalazł kolumnę `klient_id` w obu tabelach i dla każdego wiersza w `wypozyczenia` dokleił odpowiadające mu dane z `klienci`. Rezultat: **60 wierszy × 11 kolumn** (6 z lewej + 5 z prawej, bez duplikatu klucza).

To jest `inner join` — domyślne zachowanie. Wiersz trafia do wyniku tylko jeśli klucz istnieje **po obu stronach**.

### Cztery typy złączeń — parametr `how`

To najważniejsza decyzja przy `merge()`: **jak traktować wiersze, które nie mają dopasowania w drugiej tabeli?**

| `how=` | Co zachowuje | Kiedy używać |
|--------|--------------|--------------|
| `'inner'` (domyślny) | Tylko wiersze z pasującym kluczem w **obu** tabelach | Standard — szukasz kompletnych rekordów |
| `'left'` | Wszystko z lewej + dopasowania z prawej | Zachowujesz wszystkie zamówienia, nawet bez danych klienta |
| `'right'` | Wszystko z prawej + dopasowania z lewej | Zachowujesz wszystkich klientów, nawet bez zamówień |
| `'outer'` | Unia — wszystko z obu tabel | Audyt danych — chcesz widzieć wszystko |

**Wizualizacja:**

```
           inner              left              right             outer
         (A ∩ B)         (A + A∩B)          (A∩B + B)          (A ∪ B)

         ┌───┐             ┌─────┐             ┌─────┐           ┌───────┐
       A │ ∩ │ B         A │ ∩ │ B           A │ ∩ │ B         A │ ∩ │ B │
         └───┘             └─────┘             └─────┘           └───────┘
          ███              ████░░              ░░████             ████████
```

In [ ]:
# INNER — tylko wspólne klucze (domyślny)
inner = wypozyczenia.merge(klienci, on='klient_id', how='inner')
print(f"INNER: {inner.shape[0]} wierszy (zamówienia z poprawnym klientem)")

In [ ]:
# LEFT — zachowaj wszystkie wypożyczenia, nawet jeśli klienta nie ma
# Dodaj testowe wypożyczenie z nieistniejącym klientem (id=999)
wypoz_test = pd.concat([
    wypozyczenia,
    pd.DataFrame({'wypoz_id':[9999], 'klient_id':[999], 'auto_id':[1],
                   'data_od':[pd.Timestamp('2026-01-15')], 'dni':[3], 'stacja':['Warszawa']})
], ignore_index=True)

left = wypoz_test.merge(klienci, on='klient_id', how='left')
print(f"LEFT: {left.shape[0]} wierszy (wszystkie wypożyczenia, z NaN gdzie brak klienta)")
left.tail(2)

Zauważ: ostatni wiersz (id 9999) ma `NaN` w kolumnach z tabeli `klienci` — bo klient 999 nie istnieje w bazie. To jest **standardowy sygnał** problemu w danych: *„mam zamówienie od klienta, którego nie ma w bazie"*. W prawdziwych projektach pierwsze co robisz z danymi — sprawdzasz takie anomalie.

In [ ]:
# RIGHT — zachowaj wszystkich klientów, nawet jeśli nie wypożyczali
right = wypozyczenia.merge(klienci, on='klient_id', how='right')
klienci_bez_wypozycz = right[right['wypoz_id'].isna()]
print(f"RIGHT: {right.shape[0]} wierszy")
print(f"Klientów bez wypożyczenia: {klienci_bez_wypozycz.shape[0]}")
klienci_bez_wypozycz[['klient_id', 'imie', 'nazwisko']].head()

In [ ]:
# OUTER — unia: wszystko z obu tabel
outer = wypoz_test.merge(klienci, on='klient_id', how='outer')
print(f"OUTER: {outer.shape[0]} wierszy")
print(f"  - z wypoz_id i imieniem:  {outer.dropna(subset=['wypoz_id','imie']).shape[0]}")
print(f"  - tylko wypoz_id (bez klienta): {outer[outer['imie'].isna()].shape[0]}")
print(f"  - tylko klient (bez wypożyczeń): {outer[outer['wypoz_id'].isna()].shape[0]}")

### Parametr `indicator=True` — audyt dopasowań

Pandas udostępnia **specjalny trik diagnostyczny**: jeśli dodasz `indicator=True`, w wyniku pojawi się kolumna `_merge` z wartościami `both` / `left_only` / `right_only`. Idealne do pierwszej analizy jakości danych.

In [ ]:
# Merge z indicator=True — widzimy skąd pochodzi każdy wiersz
audyt = wypoz_test.merge(klienci, on='klient_id', how='outer', indicator=True)
print("Rozkład źródeł:")
print(audyt['_merge'].value_counts())

### Kiedy co używać?

- **`inner`** — gdy chcesz tylko kompletne rekordy. *„Pokaż mi wypożyczenia wraz z danymi klientów i samochodów."*
- **`left`** — gdy **lewa tabela jest ważniejsza**. *„Pokaż mi wszystkie wypożyczenia, a jeśli klient został usunięty — po prostu zostaw puste pola."*
- **`outer`** — do **audytu danych**. *„Znajdź klientów bez wypożyczeń i wypożyczenia bez klientów."*

**Wskazówka praktyczna:** W 90% przypadków używasz `inner` albo `left`. `outer` służy głównie do diagnostyki jakości danych.

### Merge po kluczach o różnych nazwach

Co jeśli klucz nazywa się inaczej w każdej tabeli? Często się zdarza — np. `klient_id` w jednym systemie, `customer_id` w drugim. Użyj wtedy parametrów `left_on` i `right_on`:

In [ ]:
# Symulacja: tabela z innego systemu ma kolumnę customer_id zamiast klient_id
klienci_alt = klienci.rename(columns={'klient_id': 'customer_id'})
merged_alt = wypozyczenia.merge(klienci_alt, left_on='klient_id', right_on='customer_id')
print(f"Merged: {merged_alt.shape}")
print(f"Kolumny klucza (obie zachowane): ['klient_id', 'customer_id']")
merged_alt[['klient_id', 'customer_id', 'imie']].head(3)

**Uwaga:** przy `left_on/right_on` obie kolumny klucza zostają w wyniku (bo mają różne nazwy). Najczęściej jedną usuwasz: `.drop(columns='customer_id')`.

---

## 3. Merge łańcuchowy — 3 tabele

Bardzo często potrzebujesz połączyć **więcej niż dwie tabele**. Pandas ułatwia to przez **łańcuchowe wywołania** — dokładasz kolejne `.merge()` jedno po drugim.

In [ ]:
# Łączymy trzy tabele w łańcuchu: wypożyczenia + klienci + samochody
pelne = (
    wypozyczenia
    .merge(klienci, on='klient_id')
    .merge(samochody, on='auto_id')
)
print(f"Pełna tabela: {pelne.shape}")
pelne.head()

**Co daje pełne złączenie?** Teraz każdy wiersz zawiera **komplet informacji**: kto wypożyczył (imię, nazwisko, miasto, segment), jaki samochód (marka, model, segment_auto), ile dni, ile zł — wszystko w jednej tabeli. **Teraz zaczyna się analiza.**

In [ ]:
# Dodajmy kolumny wyliczane — to jest moc pełnego złączenia
pelne['przychod'] = pelne['dni'] * pelne['cena_dzien']
pelne['miesiac']  = pelne['data_od'].dt.month
pelne['auto_opis'] = pelne['marka'] + ' ' + pelne['model']
print(f"Po dodaniu 3 kolumn: {pelne.shape}")
pelne[['wypoz_id', 'imie', 'miasto', 'auto_opis', 'dni', 'przychod', 'miesiac']].head()

In [ ]:
# Szybka analiza — co możemy zrobić z połączonymi danymi
print(f"Łączny przychód:   {pelne['przychod'].sum()} zł")
print(f"Średni czas wypożyczenia: {pelne['dni'].mean():.1f} dni")
print(f"Średni przychód na transakcję: {pelne['przychod'].mean():.0f} zł")

> *"The test of all knowledge is experiment. Experiment is the sole judge of scientific truth."*
> *(„Testem każdej wiedzy jest eksperyment. Eksperyment to jedyny sędzia prawdy naukowej.")*
> — Richard Feynman

Analiza danych to eksperyment. Nie wierzysz średnim — sprawdzasz rozkład. Nie wierzysz intuicji — sprawdzasz liczby. W kolejnych sekcjach zamiast pytać *„ile łącznie"* zaczniemy pytać *„ile per kategoria"* — i tu wchodzi `groupby`.

### Pułapka: duplikaty po obu stronach

Gdy klucz ma **duplikaty po obu stronach** merge, wynik eksploduje: każda para wierszy tworzy nowy rekord (iloczyn kartezjański). W naszych danych `klient_id` w tabeli `klienci` jest unikalny — ale w `wypozyczenia` jeden klient może wypożyczać wielokrotnie. To **normalny wzorzec** (many-to-one). Problem pojawia się dopiero gdy oba boki są „many".

In [ ]:
# Sprawdzenie duplikatów klucza PRZED merge — dobry nawyk
print(f"Duplikaty klient_id w klienci:     {klienci['klient_id'].duplicated().sum()}")
print(f"Duplikaty klient_id w wypozyczenia: {wypozyczenia['klient_id'].duplicated().sum()}")
print(f"(0 po stronie klientów — merge bezpieczny, many-to-one)")

---

## 4. `concat()` — sklejanie DataFrame

`merge` łączy po **kluczu** (po wartościach kolumny). `concat` łączy po **pozycji** — skleja DataFrame'y w dół (nowe wiersze) lub w bok (nowe kolumny).

**Kiedy `merge`, kiedy `concat`?**

| Sytuacja | Narzędzie |
|----------|-----------|
| Mam dwie różne tabele z kluczem łączącym | `merge` |
| Mam **tę samą strukturę** kolumn, ale z różnych okresów/źródeł | `concat` (axis=0) |
| Mam nowe kolumny dla tych samych wierszy | `concat` (axis=1) lub `merge` |

In [ ]:
# Scenariusz: dane z dwóch kwartałów — ta sama struktura, inne wiersze
q1 = wypozyczenia[wypozyczenia['data_od'] < '2026-02-01'].copy()
q1['kwartal'] = 'Q1'
q2 = wypozyczenia[(wypozyczenia['data_od'] >= '2026-02-01') & (wypozyczenia['data_od'] < '2026-03-01')].copy()
q2['kwartal'] = 'Q2'

print(f"Q1: {q1.shape[0]} wypożyczeń")
print(f"Q2: {q2.shape[0]} wypożyczeń")

In [ ]:
# concat axis=0 (wiersze, domyślne) — skleja w dół
polaczone = pd.concat([q1, q2], ignore_index=True)
print(f"Połączone: {polaczone.shape[0]} wierszy")
print(f"Pierwsze 2 z q1, ostatnie 2 z q2:")
polaczone[['wypoz_id', 'data_od', 'kwartal']].iloc[[0, 1, -2, -1]]

Parametr `ignore_index=True` **resetuje indeks** (nowy 0..n-1). Bez niego zachowywałyby się stare indeksy z obu DataFrame'ów — i miałbyś duplikaty.

In [ ]:
# concat z keys — zachowuje informację o źródle jako MultiIndex
polaczone_keys = pd.concat([q1, q2], keys=['Q1_dane', 'Q2_dane'])
print(f"MultiIndex (pierwsze 2 wiersze):")
polaczone_keys[['wypoz_id', 'data_od']].head(2)

In [ ]:
# concat axis=1 — dodawanie nowych kolumn (po pozycji)
# Uwaga: wymaga tego samego indeksu w obu DataFrame
dane1 = klienci[['klient_id', 'imie']].head(5)
dane2 = klienci[['miasto', 'segment']].head(5)
polaczone_bok = pd.concat([dane1, dane2], axis=1)
polaczone_bok

**Uwaga:** `concat` ma jedną pułapkę — dokleja po **pozycji** (indeksie), nie po wartości klucza. Jeśli wiersze się nie zgadzają kolejnością, dostaniesz niechciane `NaN`. Gdy masz klucz łączący — zawsze `merge`.

---

## 5. `groupby()` — grupowanie i agregacja

`groupby` odpowiada na pytania typu *„ile X **per** Y?"*. Dzieli DataFrame na grupy według wartości w kolumnie i wykonuje operację na każdej grupie osobno.

**Analogia Excela:** `groupby` + `.sum()` to polskie **SUMY.JEŻELI()**. Tylko bez klikania 100 razy.

```mermaid
graph TD
    A["DataFrame: 60 wypozyczen"] -->|"groupby miasto"| B["Grupa: Warszawa"]
    A --> C["Grupa: Krakow"]
    A --> D["Grupa: Gdansk"]
    B -->|"sum"| E["suma przychodu Warszawy"]
    C -->|"sum"| F["suma przychodu Krakowa"]
    D -->|"sum"| G["suma przychodu Gdanska"]
```

Workflow `groupby` to **Split → Apply → Combine**:
1. **Split** — podziel na grupy
2. **Apply** — policz funkcję per grupa
3. **Combine** — złóż w jeden wynik

In [ ]:
# Podstawowy groupby — suma przychodu per miasto klienta
przychod_per_miasto = pelne.groupby('miasto', observed=True)['przychod'].sum()
print("Przychód per miasto klienta:")
print(przychod_per_miasto.sort_values(ascending=False))

**Wyjaśnienie kroków:**
1. `pelne.groupby('miasto', observed=True)` — dzielimy DataFrame na grupy wg kolumny `miasto`
2. `['przychod']` — interesuje nas jedna kolumna: `przychod`
3. `.sum()` — w każdej grupie sumujemy
4. `.sort_values(ascending=False)` — malejąco

**Parametr `observed=True`** — warto go zawsze dodawać przy grupowaniu po kolumnie kategorycznej. Bez niego Pandas generuje ostrzeżenia i tworzy puste grupy dla niewystępujących kategorii.

In [ ]:
# Wiele funkcji agregujących naraz
statystyki = pelne.groupby('miasto', observed=True)['przychod'].agg(['sum', 'mean', 'count'])
print("Statystyki przychodów per miasto:")
statystyki.round(2)

### `.agg()` z dictem — różne funkcje per kolumna

Kiedy chcesz różne statystyki dla różnych kolumn — podaj `agg()` słownik:

In [ ]:
# Różne funkcje dla różnych kolumn
summary = pelne.groupby('miasto', observed=True).agg({
    'dni':      'mean',      # średnia długość wypożyczenia
    'przychod': 'sum',       # łączny przychód
    'wypoz_id': 'count'      # liczba transakcji
})
summary.columns = ['srednie_dni', 'laczny_przychod', 'liczba_transakcji']
summary.round(1).sort_values('laczny_przychod', ascending=False)

### Named aggregation — czytelne nazwy

Najnowszy i najczystszy sposób (od Pandas 0.25+). Używa krotek `(kolumna, funkcja)` jako wartości:

In [ ]:
# Named aggregation — jawne nazwy kolumn wynikowych
raport = pelne.groupby('segment_auto', observed=True).agg(
    liczba_wypoz=('wypoz_id', 'count'),
    laczne_dni=('dni', 'sum'),
    sredni_przychod=('przychod', 'mean'),
    max_przychod=('przychod', 'max')
).round(2)
raport.sort_values('sredni_przychod', ascending=False)

**Co daje named aggregation?**
- Czytelne nazwy kolumn wynikowych — nie musisz później robić `.columns = [...]`
- Jawność: od razu widać co, z czego, jaką funkcją
- To **oficjalnie polecany styl** w nowoczesnym Pandas

### groupby po wielu kolumnach

Możesz grupować po więcej niż jednej kolumnie — dostajesz wtedy wyniki „na przecięciu".

In [ ]:
# Grupowanie po dwóch wymiarach: segment klienta × segment auta
przychod_2d = pelne.groupby(
    ['segment', 'segment_auto'], observed=True
)['przychod'].sum().round(0)
print("Przychód: segment klienta × segment auta")
przychod_2d

Zauważ: wynik ma **MultiIndex** (dwa poziomy indeksu). To często niewygodne — dlatego w następnej sekcji pokażemy `pivot_table`, który to samo zrobi w czytelnej formie 2D.

### `.transform()` — wartość grupowa w każdym wierszu

`groupby().transform()` zwraca DataFrame tej samej długości co oryginał — przypisuje wartość grupową każdemu wierszowi. Idealne do liczenia udziału procentowego.

In [ ]:
# Przychód każdej transakcji + łączny przychód miasta + udział %
pelne['przychod_miasta'] = pelne.groupby('miasto', observed=True)['przychod'].transform('sum')
pelne['udzial_pct'] = (pelne['przychod'] / pelne['przychod_miasta'] * 100).round(1)
pelne[['wypoz_id', 'miasto', 'przychod', 'przychod_miasta', 'udzial_pct']].head()

**Różnica `agg` vs `transform`:**

| Metoda | Długość wyniku | Do czego |
|--------|----------------|----------|
| `.agg()` | **Jedna wartość per grupa** (skraca) | Raporty, podsumowania |
| `.transform()` | **Ta sama długość co input** (nie skraca) | Kolumny wyliczane, udziały, normalizacja |

### Szybkie rankingi — `nlargest` i `nsmallest`

Dla „TOP N per grupa" Pandas ma gotowe metody `nlargest(n, 'kolumna')` i `nsmallest(n, 'kolumna')`.

In [ ]:
# TOP 3 najdłuższych wypożyczeń OGÓŁEM
top3_overall = pelne.nlargest(3, 'dni')[['imie', 'nazwisko', 'auto_opis', 'dni', 'przychod']]
print("TOP 3 najdłuższe wypożyczenia:")
top3_overall

In [ ]:
# TOP 2 najdroższe wypożyczenia per segment auta
top_per_segment = (
    pelne.groupby('segment_auto', observed=True, group_keys=False)
         .apply(lambda d: d.nlargest(2, 'przychod'), include_groups=False)
)
top_per_segment[['imie', 'nazwisko', 'auto_opis', 'przychod']].head(6)

---

## 6. `pivot_table()` — tabela przestawna

**To jest dokładnie to samo, co tabele przestawne w Excelu** — tylko w Pythonie. Agregujesz dane po **dwóch wymiarach jednocześnie** (np. wiersze = miesiąc, kolumny = segment, wartości = przychód).

Składnia:
```python
pd.pivot_table(df, index='wiersze', columns='kolumny', values='wartosc', aggfunc='sum')
```

In [ ]:
# Tabela przestawna: segment_auto (wiersze) × miesiąc (kolumny), wartości = przychód
pivot = pelne.pivot_table(
    index='segment_auto',
    columns='miesiac',
    values='przychod',
    aggfunc='sum',
    fill_value=0,
    observed=True
)
pivot.columns = [f"Miesiac_{c}" for c in pivot.columns]
print("Przychód: segment auto × miesiąc")
pivot.round(0)

**Ten sam wynik, który przed chwilą dawał `groupby(['segment_auto','miesiac'])`, jest teraz w czytelnej formie 2D.** `pivot_table` to `groupby` + `unstack` w jednym wywołaniu.

### Ciekawostka: `pivot` vs `pivot_table`

Pandas ma **dwie podobne funkcje**: `df.pivot()` i `df.pivot_table()`. Różnią się jedną kluczową rzeczą:

| | `pivot` | `pivot_table` |
|---|---------|---------------|
| Zachowanie przy duplikatach | **Błąd**! | **Agreguje** (domyślnie suma) |
| Parametr `aggfunc` | Nie ma | Jest (`'sum'`, `'mean'`, `'count'`, lista) |
| Parametr `margins` (sumy brzegowe) | Nie ma | Jest |

**Zasada praktyczna:** używaj zawsze `pivot_table()` — działa w każdej sytuacji.

In [ ]:
# pivot_table z margins=True — dodaje sumy brzegowe (jak w tabeli przestawnej Excel)
pivot_margins = pelne.pivot_table(
    index='segment',
    columns='segment_auto',
    values='przychod',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='RAZEM',
    observed=True
)
print("Przychód z sumami brzegowymi:")
pivot_margins.round(0)

Wiersz i kolumna „RAZEM" pokazują sumy brzegowe — dokładnie jak w tabeli przestawnej w Excelu z włączoną opcją „Grand totals". Studenci często nie zdają sobie sprawy, że Excel pod spodem robi dokładnie to samo co Pandas.

In [ ]:
# pivot_table z WIELOMA funkcjami agregującymi
pivot_multi = pelne.pivot_table(
    index='miasto',
    columns='segment_auto',
    values='przychod',
    aggfunc=['sum', 'count'],
    fill_value=0,
    observed=True
)
print("Suma przychodu i liczba transakcji:")
pivot_multi.round(0)

Wynik ma **MultiIndex na kolumnach** — wyższy poziom to typ statystyki (`sum` / `count`), niższy to `segment_auto`. Do konkretnej sekcji odwołujesz się tak: `pivot_multi['sum']['premium']`.

---

## 7. `crosstab()` — liczebności (tablica kontyngencji)

`crosstab` to **specjalny przypadek `pivot_table`** — zlicza wystąpienia kombinacji dwóch kolumn. Używaj gdy interesują Cię **liczby**, nie sumy wartości.

In [ ]:
# crosstab: ile transakcji per kombinacja segmentu klienta i segmentu auta
ct = pd.crosstab(pelne['segment'], pelne['segment_auto'])
print("Liczba wypożyczeń: segment klienta × segment auta")
ct

In [ ]:
# crosstab z normalize — procenty zamiast liczb
ct_pct = pd.crosstab(
    pelne['segment'],
    pelne['segment_auto'],
    normalize='index'   # procent wewnątrz każdego wiersza
) * 100
print("Rozkład procentowy (% w wierszu):")
ct_pct.round(1)

**Parametr `normalize`** ma trzy tryby:
- `'index'` — procenty sumują się do 100% w każdym wierszu (rozkład zależny od lewej zmiennej)
- `'columns'` — procenty sumują się do 100% w każdej kolumnie
- `'all'` / `True` — procenty sumują się do 100% w całej tabeli

### `pivot_table` vs `crosstab` — kiedy co?

| Zastosowanie | Używaj |
|--------------|--------|
| **Liczenie wystąpień** (ile klientów w mieście?) | `crosstab` |
| **Agregowanie wartości** (ile zł per miasto?) | `pivot_table` |
| **Procenty rozkładu** (jaki % klientów...?) | `crosstab(normalize=...)` |
| **Skomplikowane funkcje** (różne per kolumna) | `pivot_table` z dict w aggfunc |

> **Ciekawostka:** Pandas został stworzony w 2008 r. przez **Wesa McKinneya** — analityka w funduszu hedgingowym AQR. Potrzebował narzędzia do analizy szeregów czasowych (kursy akcji, obligacji). `pivot_table` była jedną z pierwszych funkcji — bo „analitycy finansowi żyją tabelami przestawnymi". Dziś Pandas jest używany przez Netflix, NASA, Spotify i każdy poważny bank na świecie.

---

## 8. Mini-ćwiczenia na wykładzie

Trzy krótkie zadania — sprawdzenie, czy omówione metody są zrozumiałe. Czas: ~10 minut.

### Zadanie A — TOP-3 miasta wg przychodu

Posortuj miasta po łącznym przychodzie z wypożyczeń (malejąco) i wypisz pierwsze 3. Użyj `groupby` + `sum` + `sort_values`.

### Zadanie B — Średnia długość wypożyczenia per segment auta

Dla każdego `segment_auto` policz średnią liczbę dni wypożyczenia. Wynik zaokrąglij do 1 miejsca po przecinku.

### Zadanie C — Pivot miesiąc × segment auta

Zbuduj `pivot_table` gdzie wiersze to `miesiac`, kolumny to `segment_auto`, a wartości to **liczba** wypożyczeń (`aggfunc='count'`).

---

Spróbuj sam, zanim spojrzysz na rozwiązania poniżej.

In [ ]:
# Rozwiązanie A: TOP-3 miasta wg przychodu
top3_miast = pelne.groupby('miasto', observed=True)['przychod'].sum().sort_values(ascending=False).head(3)
print("TOP-3 miasta wg przychodu:")
print(top3_miast)

In [ ]:
# Rozwiązanie B: Średnia długość wypożyczenia per segment auta
srednie_dni = pelne.groupby('segment_auto', observed=True)['dni'].mean().round(1).sort_values(ascending=False)
print("Średnia długość wypożyczenia per segment:")
print(srednie_dni)

In [ ]:
# Rozwiązanie C: pivot miesiąc × segment_auto, liczba wypożyczeń
pivot_liczba = pelne.pivot_table(
    index='miesiac',
    columns='segment_auto',
    values='wypoz_id',
    aggfunc='count',
    fill_value=0,
    observed=True
)
print("Liczba wypożyczeń: miesiąc × segment auta")
pivot_liczba

**Pytanie refleksyjne:** Każde z tych zadań można rozwiązać na **co najmniej dwa sposoby** — np. pivot_table z aggfunc='count' zamiast groupby + size. W pracy zawodowej ważne jest nie *„jedno słuszne rozwiązanie"*, ale: *czy Twoje rozwiązanie jest czytelne?*

Jeśli kolega/koleżanka z zespołu otworzy Twój kod za tydzień — zrozumie go w 30 sekund?

---

## Podgląd laboratorium

Na laboratorium przećwiczycie `merge`, `groupby` i `pivot_table` na **nowym datasecie**: **e-sklep książek** — autorzy × książki × zamówienia. Inna branża, ten sam schemat analityczny.

**Start — 3 komendy:**
```
cd C:\Users\student\python2
.venv\Scripts\Activate.ps1
code .
```

**Ćwiczenia na labie (90 min):**

- **Ćwiczenie 1** *(20 min)* — merge dwóch tabel (książki + autorzy), cztery typy `how`, analiza wyniku
- **Ćwiczenie 2** *(20 min)* — merge łańcuchowy (3 tabele: autorzy + książki + zamówienia), dodawanie kolumn wyliczanych
- **Ćwiczenie 3** *(30 min)* — `groupby` + `.agg()` — statystyki per kategoria i per autor; named aggregation
- **Ćwiczenie 4** *(20 min)* — `pivot_table` (kategoria × miesiąc) + `crosstab` (rozkłady procentowe); raport biznesowy

Utworzysz notebook `lab08_pandas_merge.ipynb`. Commity po każdym ćwiczeniu. Zero niespodzianek — wszystkie komendy znasz z dzisiejszego wykładu.

---

## Na koniec — ściąga: „Chcę zrobić X"

### `merge` — łączenie tabel

| Cel | Kod |
|-----|-----|
| Inner join po jednym kluczu | `a.merge(b, on='klucz')` |
| Left join (zachowaj lewą) | `a.merge(b, on='klucz', how='left')` |
| Outer join (unia) | `a.merge(b, on='klucz', how='outer')` |
| Klucze mają różne nazwy | `a.merge(b, left_on='id_a', right_on='id_b')` |
| Łańcuchowy merge 3 tabel | `a.merge(b, on='k1').merge(c, on='k2')` |
| Audyt dopasowań | `a.merge(b, on='k', how='outer', indicator=True)` |
| Liczenie braków po lewej | `merged[merged['kol_z_prawej'].isna()]` |

### `concat` — sklejanie

| Cel | Kod |
|-----|-----|
| Sklej w dół (nowe wiersze) | `pd.concat([df1, df2], ignore_index=True)` |
| Sklej w bok (nowe kolumny) | `pd.concat([df1, df2], axis=1)` |
| Oznacz źródło | `pd.concat([df1, df2], keys=['A', 'B'])` |

### `groupby` + `agg`

| Cel | Kod |
|-----|-----|
| Suma per grupa | `df.groupby('kol')['wartosc'].sum()` |
| Wiele funkcji | `df.groupby('kol')['wartosc'].agg(['sum', 'mean', 'count'])` |
| Różne funkcje per kolumna | `df.groupby('kol').agg({'a':'sum', 'b':'mean'})` |
| Named aggregation | `df.groupby('kol').agg(suma=('a','sum'), srednia=('b','mean'))` |
| Grupa po wielu kolumnach | `df.groupby(['kol1','kol2'])['wartosc'].sum()` |
| Wartość grupowa w każdym wierszu | `df.groupby('kol')['wartosc'].transform('sum')` |
| Udział procentowy | `df['w'] / df.groupby('kol')['w'].transform('sum') * 100` |
| TOP N per grupa | `df.groupby('kol', group_keys=False).apply(lambda x: x.nlargest(n, 'wart'))` |

### `pivot_table` i `crosstab`

| Cel | Kod |
|-----|-----|
| Tabela przestawna | `pd.pivot_table(df, index='w', columns='k', values='v', aggfunc='sum')` |
| Z sumami brzegowymi | `pivot_table(..., margins=True, margins_name='RAZEM')` |
| Wiele funkcji | `pivot_table(..., aggfunc=['sum','mean'])` |
| Liczba wystąpień | `pd.crosstab(df['a'], df['b'])` |
| Rozkład procentowy | `pd.crosstab(a, b, normalize='index') * 100` |

### Pułapki — czego unikać

- **Duplikaty kluczy** — gdy po obu stronach merge są powtórki, wynik eksploduje (kartezjański). Sprawdzaj: `a['klucz'].duplicated().sum()`.
- **Różne typy kluczy** — `int` vs `str` nie złączą się nigdy. Zunifikuj typ przed merge.
- **observed=True** — gdy grupujesz po kolumnie kategorycznej (`pd.Categorical`), dodawaj `observed=True`, aby uniknąć ostrzeżeń i pustych grup.
- **`NaN` w kluczu** — `NaN` nie łączy się z `NaN`. Najpierw wyczyść braki.
- **`pivot` vs `pivot_table`** — używaj zawsze `pivot_table` (agreguje duplikaty).

## Źródła

| Źródło | Opis | Link |
|--------|------|------|
| Pandas — `merge` | Oficjalna dokumentacja `DataFrame.merge` | pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html |
| Pandas — Merge, join, concat | User guide (pełny rozdział) | pandas.pydata.org/docs/user_guide/merging.html |
| Pandas — `groupby` | User guide — Split-Apply-Combine | pandas.pydata.org/docs/user_guide/groupby.html |
| Pandas — `pivot_table` | Dokumentacja funkcji | pandas.pydata.org/docs/reference/api/pandas.pivot_table.html |
| Pandas — reshaping | Reshape & pivot — szczegóły | pandas.pydata.org/docs/user_guide/reshaping.html |
| Wes McKinney — Python for Data Analysis | Rozdział 8: Data Wrangling (merge, groupby, reshape) | wesmckinney.com/book/data-wrangling.html |

---

> *„If you can't explain it simply, you don't understand it well enough."*
> *(„Jeśli nie potrafisz tego wyjaśnić prosto, nie rozumiesz tego wystarczająco dobrze.")*
> — Richard Feynman

**Za tydzień:** Matplotlib — wizualizacja danych. Wszystkie tabele, które dziś budowaliśmy, zamienimy w wykresy, które „mówią same z siebie".